# Backpropagation at Research Level
## Theory, matrix calculus, Jacobians, worked examples, and code

This notebook is a **research-grade study note** on backpropagation. It is designed to be useful not just for implementing neural networks, but also for reading papers, understanding autodiff, and reasoning about gradient flow in deep systems.

It includes:

1. Conceptual overview of backpropagation  
2. Scalar chain rule to multivariable chain rule  
3. Jacobian matrices and Jacobian-vector products  
4. Gradients, Jacobians, Hessians, and notation discipline  
5. Backpropagation for a single neuron  
6. Backpropagation for multilayer perceptrons  
7. Matrix-calculus derivation in mini-batch form  
8. Fully worked numerical example by hand  
9. Manual NumPy implementation  
10. Gradient checking with finite differences  
11. Why backprop uses **vector-Jacobian products**, not full Jacobians  
12. PyTorch autograd verification  
13. Vanishing gradients, ReLU, and geometric interpretation


## Notation

We will use:

- Input: $x \in \mathbb{R}^{d_0}$
- Layer $l$ pre-activation: $z^{[l]} \in \mathbb{R}^{d_l}$
- Layer $l$ activation: $a^{[l]} \in \mathbb{R}^{d_l}$
- Parameters: $W^{[l]} \in \mathbb{R}^{d_l \times d_{l-1}}$, $b^{[l]} \in \mathbb{R}^{d_l}$
- Loss for one example: $L$
- Batch loss: $J$

We set

$$
a^{[0]} = x.
$$

For a feedforward network,

$$
z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]},
$$

$$
a^{[l]} = \phi^{[l]}(z^{[l]}).
$$

The output layer is $a^{[L]}$, and the loss is

$$
L = \mathcal{L}(a^{[L]}, y).
$$


# 1. Why backpropagation exists

Training a neural network means optimizing parameters $\theta$ to reduce a loss:

$$
\min_\theta J(\theta).
$$

Gradient-based optimization requires the gradient

$$
\nabla_\theta J.
$$

Each parameter typically influences the loss only **indirectly**, through many intermediate computations. For a simple chain

$$
w \rightarrow z \rightarrow a \rightarrow L,
$$

the derivative is

$$
\frac{\partial L}{\partial w}
=
\frac{\partial L}{\partial a}
\frac{\partial a}{\partial z}
\frac{\partial z}{\partial w}.
$$

A deep neural network is just a much larger nested composition of such computations.

Backpropagation is **not** a different principle from the chain rule. It is an **efficient organization** of the chain rule on a computational graph.


# 2. Derivative, gradient, Jacobian, Hessian

If $f: \mathbb{R} \to \mathbb{R}$, then the derivative is

$$
\frac{df}{dx}.
$$

If $f: \mathbb{R}^n \to \mathbb{R}$, the gradient is

$$
\nabla_x f
=
\begin{bmatrix}
\frac{\partial f}{\partial x_1} \\
\frac{\partial f}{\partial x_2} \\
\vdots \\
\frac{\partial f}{\partial x_n}
\end{bmatrix}.
$$

If $f: \mathbb{R}^n \to \mathbb{R}^m$, the Jacobian is

$$
J_f(x)
=
\frac{\partial f}{\partial x}
=
\begin{bmatrix}
\frac{\partial f_1}{\partial x_1} & \cdots & \frac{\partial f_1}{\partial x_n} \\
\frac{\partial f_2}{\partial x_1} & \cdots & \frac{\partial f_2}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\frac{\partial f_m}{\partial x_1} & \cdots & \frac{\partial f_m}{\partial x_n}
\end{bmatrix}.
$$

If $f: \mathbb{R}^n \to \mathbb{R}$, the Hessian is

$$
H_f(x)=\nabla_x^2 f.
$$


# 3. Jacobian intuition

Suppose

$$
y = f(x), \qquad x \in \mathbb{R}^n,\; y \in \mathbb{R}^m.
$$

The Jacobian gives the best local linear approximation near a point $x$:

$$
f(x + \Delta x) \approx f(x) + J_f(x)\,\Delta x.
$$

So the Jacobian tells us how the function **locally stretches, rotates, shears, and compresses** input space.


# 4. Chain rule in Jacobian form

Let

$$
g: \mathbb{R}^n \to \mathbb{R}^p,\qquad
f: \mathbb{R}^p \to \mathbb{R}^m,
$$

and define $h = f \circ g$.

Then

$$
J_h(x) = J_f(g(x)) \, J_g(x).
$$

For neural networks, the forward computation is a repeated composition of layer maps.

In practice, backpropagation does **not** form full Jacobians. It computes **vector-Jacobian products** efficiently.


# 5. Example Jacobian

Let

$$
f(x_1, x_2)
=
\begin{bmatrix}
x_1^2 + x_2 \\
x_1 x_2
\end{bmatrix}.
$$

Then

$$
J_f(x_1, x_2)
=
\begin{bmatrix}
2x_1 & 1 \\
x_2 & x_1
\end{bmatrix}.
$$


In [ ]:
import numpy as np

def f_vec(x):
    x1, x2 = x
    return np.array([x1**2 + x2, x1*x2], dtype=float)

x = np.array([2.0, 3.0])
J = np.array([[2*x[0], 1.0],
              [x[1], x[0]]], dtype=float)

dx = np.array([1e-3, -2e-3], dtype=float)
approx = J @ dx
actual = f_vec(x + dx) - f_vec(x)

print("Jacobian at x =")
print(J)
print("\nJ dx =")
print(approx)
print("\nf(x+dx)-f(x) =")
print(actual)


# 6. Jacobians of common neural-network operations

## Linear map

For

$$
z = Wx + b,
$$

the Jacobian with respect to $x$ is

$$
\frac{\partial z}{\partial x} = W.
$$

## Elementwise activation

If

$$
a_i = \phi(z_i),
$$

then

$$
\frac{\partial a}{\partial z}
=
\operatorname{diag}\left(\phi'(z_1), \ldots, \phi'(z_n)\right).
$$

## Sigmoid

$$
\sigma(z) = \frac{1}{1+e^{-z}},
\qquad
\sigma'(z)=\sigma(z)(1-\sigma(z)).
$$

## Softmax Jacobian

If $s = \operatorname{softmax}(z)$, then

$$
J_{\text{softmax}}(z)
=
\operatorname{diag}(s) - s s^\top.
$$


# 7. Single neuron derivation

Consider

$$
z = wx + b,
\qquad
a = \phi(z),
\qquad
L = \frac{1}{2}(a-y)^2.
$$

Then

$$
\frac{\partial L}{\partial w}
=
\frac{\partial L}{\partial a}
\frac{\partial a}{\partial z}
\frac{\partial z}{\partial w}
=
(a-y)\phi'(z)x,
$$

and

$$
\frac{\partial L}{\partial b}
=
(a-y)\phi'(z).
$$

Define

$$
\delta = \frac{\partial L}{\partial z} = (a-y)\phi'(z).
$$

Then

$$
\frac{\partial L}{\partial w} = \delta x,
\qquad
\frac{\partial L}{\partial b} = \delta.
$$


# 8. Multilayer network and delta recursion

For layers $1,\dots,L$:

$$
z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]},
\qquad
a^{[l]} = \phi^{[l]}(z^{[l]}).
$$

Define

$$
\delta^{[l]} := \frac{\partial L}{\partial z^{[l]}}.
$$

## Output layer

$$
\delta^{[L]}
=
\frac{\partial L}{\partial a^{[L]}}
\odot
\phi^{[L]\,'}(z^{[L]}).
$$

## Hidden layers

$$
\delta^{[l]}
=
\left(W^{[l+1]\top}\delta^{[l+1]}\right)
\odot
\phi^{[l]\,'}(z^{[l]}).
$$

## Parameter gradients

$$
\frac{\partial L}{\partial W^{[l]}}
=
\delta^{[l]} a^{[l-1]\top},
\qquad
\frac{\partial L}{\partial b^{[l]}}
=
\delta^{[l]}.
$$


# 9. Mini-batch matrix calculus derivation

Arrange a batch of $m$ examples as columns:

$$
A^{[0]} = X \in \mathbb{R}^{d_0 \times m}.
$$

For each layer:

$$
Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]} \mathbf{1}^\top,
$$

$$
A^{[l]} = \phi^{[l]}(Z^{[l]}).
$$

Let

$$
J = \frac{1}{m}\sum_{k=1}^m L^{(k)}.
$$

Define

$$
D^{[l]} := \frac{\partial J}{\partial Z^{[l]}}.
$$

Then

$$
D^{[L]}
=
\frac{\partial J}{\partial A^{[L]}}
\odot
\phi^{[L]\,'}(Z^{[L]}),
$$

$$
D^{[l]}
=
\left(W^{[l+1]\top} D^{[l+1]}\right)
\odot
\phi^{[l]\,'}(Z^{[l]}).
$$

And the parameter gradients are

$$
dW^{[l]} = \frac{1}{m} D^{[l]} A^{[l-1]\top},
\qquad
db^{[l]} = \frac{1}{m}\sum_{k=1}^{m} D^{[l]}_{:,k}.
$$


# 10. Fully worked numerical example

We use a network with:

- input dimension: 2
- hidden layer: 2 neurons
- output layer: 1 neuron
- sigmoid activations
- mean squared error loss

Input:

$$
x =
\begin{bmatrix}
1 \\
2
\end{bmatrix},
\qquad
y = 1
$$

Parameters:

$$
W^{[1]} =
\begin{bmatrix}
0.1 & 0.2 \\
0.3 & 0.4
\end{bmatrix},
\quad
b^{[1]} =
\begin{bmatrix}
0.1 \\
0.1
\end{bmatrix}
$$

$$
W^{[2]} =
\begin{bmatrix}
0.5 & 0.6
\end{bmatrix},
\quad
b^{[2]} =
\begin{bmatrix}
0.1
\end{bmatrix}.
$$


## Forward pass

$$
z^{[1]} = W^{[1]}x + b^{[1]}
=
\begin{bmatrix}
0.6 \\
1.2
\end{bmatrix}
$$

$$
a^{[1]} = \sigma(z^{[1]})
\approx
\begin{bmatrix}
0.64565631 \\
0.76852478
\end{bmatrix}
$$

$$
z^{[2]} = W^{[2]}a^{[1]} + b^{[2]}
\approx 0.88394303
$$

$$
a^{[2]} = \sigma(z^{[2]}) \approx 0.70763783
$$

$$
L = \frac{1}{2}(a^{[2]} - y)^2 \approx 0.04273782.
$$


## Backward pass

Output delta:

$$
\delta^{[2]}
=
(a^{[2]}-y)\,a^{[2]}(1-a^{[2]})
\approx -0.06050134.
$$

Output gradients:

$$
\frac{\partial L}{\partial W^{[2]}}
=
\delta^{[2]} a^{[1]\top}
\approx
\begin{bmatrix}
-0.03905682 & -0.04649776
\end{bmatrix}
$$

$$
\frac{\partial L}{\partial b^{[2]}}
=
\delta^{[2]}
\approx -0.06050134.
$$

Hidden delta:

$$
\delta^{[1]}
=
\left(W^{[2]\top}\delta^{[2]}\right)\odot a^{[1]}(1-a^{[1]})
\approx
\begin{bmatrix}
-0.00692053 \\
-0.00645807
\end{bmatrix}.
$$

Hidden gradients:

$$
\frac{\partial L}{\partial W^{[1]}}
=
\delta^{[1]}x^\top
\approx
\begin{bmatrix}
-0.00692053 & -0.01384105 \\
-0.00645807 & -0.01291614
\end{bmatrix}
$$

$$
\frac{\partial L}{\partial b^{[1]}}
=
\delta^{[1]}.
$$


In [ ]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

x = np.array([[1.0], [2.0]])
y = np.array([[1.0]])

W1 = np.array([[0.1, 0.2],
               [0.3, 0.4]])
b1 = np.array([[0.1],
               [0.1]])

W2 = np.array([[0.5, 0.6]])
b2 = np.array([[0.1]])

z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
a2 = sigmoid(z2)
L = 0.5 * np.sum((a2 - y)**2)

delta2 = (a2 - y) * a2 * (1 - a2)
dW2 = delta2 @ a1.T
db2 = delta2

delta1 = (W2.T @ delta2) * a1 * (1 - a1)
dW1 = delta1 @ x.T
db1 = delta1

print("z1 =\n", z1)
print("a1 =\n", a1)
print("z2 =\n", z2)
print("a2 =\n", a2)
print("L =\n", L)

print("\ndelta2 =\n", delta2)
print("dW2 =\n", dW2)
print("db2 =\n", db2)

print("\ndelta1 =\n", delta1)
print("dW1 =\n", dW1)
print("db1 =\n", db1)


# 11. One gradient-descent update

If $\eta = 0.1$, then

$$
W^{[l]} \leftarrow W^{[l]} - \eta \frac{\partial L}{\partial W^{[l]}},
\qquad
b^{[l]} \leftarrow b^{[l]} - \eta \frac{\partial L}{\partial b^{[l]}}.
$$


In [ ]:
lr = 0.1

W1_new = W1 - lr * dW1
b1_new = b1 - lr * db1
W2_new = W2 - lr * dW2
b2_new = b2 - lr * db2

print("Updated W1 =\n", W1_new)
print("Updated b1 =\n", b1_new)
print("Updated W2 =\n", W2_new)
print("Updated b2 =\n", b2_new)


# 12. Why backprop does not form full Jacobians

Suppose a layer maps $\mathbb{R}^{1000} \to \mathbb{R}^{1000}$.
Its Jacobian is a $1000 \times 1000$ matrix.

Reverse-mode autodiff does **not** build this matrix explicitly.
Instead, if $u = f(v)$ and the upstream gradient is

$$
g_u = \frac{\partial L}{\partial u},
$$

then the needed quantity is

$$
g_v = \frac{\partial L}{\partial v} = J_f(v)^\top g_u.
$$

So each backward step is a **Jacobian-transpose-vector product**.

That is one of the main reasons backpropagation is computationally feasible for large models.


# 13. Gradient checking

For a scalar parameter $\theta$,

$$
\frac{\partial L}{\partial \theta}
\approx
\frac{L(\theta+\varepsilon)-L(\theta-\varepsilon)}{2\varepsilon}.
$$

This central-difference estimate is a standard way to verify manual backpropagation code.


In [ ]:
def forward_loss(W1, b1, W2, b2, x, y):
    z1 = W1 @ x + b1
    a1 = 1/(1+np.exp(-z1))
    z2 = W2 @ a1 + b2
    a2 = 1/(1+np.exp(-z2))
    return 0.5 * np.sum((a2 - y)**2)

analytic = dW1[0, 0]
eps = 1e-5

W1_plus = W1.copy()
W1_minus = W1.copy()
W1_plus[0, 0] += eps
W1_minus[0, 0] -= eps

L_plus = forward_loss(W1_plus, b1, W2, b2, x, y)
L_minus = forward_loss(W1_minus, b1, W2, b2, x, y)

numeric = (L_plus - L_minus) / (2 * eps)

print("Analytic gradient:", analytic)
print("Numeric gradient :", numeric)
print("Absolute diff    :", abs(analytic - numeric))


# 14. Reverse-mode autodiff perspective

A neural network can be represented as a computational graph of primitive operations:

- matrix multiplication,
- addition,
- elementwise activation,
- reduction to a scalar loss.

For a primitive operation $u = f(v)$, the backward rule is

$$
\bar{v} = J_f(v)^\top \bar{u},
$$

where $\bar{u}$ denotes the adjoint, usually interpreted as $\partial L / \partial u$.

Reverse-mode autodiff traverses the graph in reverse order and accumulates these adjoints.

That is exactly what `loss.backward()` does in PyTorch.


# 15. Softmax + cross-entropy simplification

For logits $z \in \mathbb{R}^K$, softmax outputs

$$
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}},
$$

and one-hot target $y$, the cross-entropy loss is

$$
L = -\sum_{i=1}^K y_i \log p_i.
$$

Then the gradient with respect to logits simplifies to

$$
\frac{\partial L}{\partial z} = p - y.
$$

This is why many derivations write directly

$$
\delta^{[L]} = \hat{y} - y
$$

for softmax classification.


# 16. Vanishing gradients and Jacobian products

For deep compositions, gradients propagate through products of local Jacobians:

$$
\frac{\partial L}{\partial x}
=
J_1^\top J_2^\top \cdots J_L^\top \frac{\partial L}{\partial a^{[L]}}.
$$

If typical Jacobian norms are less than 1, gradients tend to shrink exponentially with depth.

That is the **vanishing gradient problem**.

If they are greater than 1, gradients may explode.


In [ ]:
import numpy as np

np.random.seed(0)
depth = 30
width = 50

sigmoid_diags = [np.diag(np.random.uniform(0.01, 0.25, size=width)) for _ in range(depth)]
relu_diags = [np.diag(np.random.binomial(1, 0.5, size=width).astype(float)) for _ in range(depth)]

sigmoid_product = np.eye(width)
relu_product = np.eye(width)

for D in sigmoid_diags:
    sigmoid_product = D @ sigmoid_product

for D in relu_diags:
    relu_product = D @ relu_product

print("||product of sigmoid-like Jacobians||_2 ≈", np.linalg.norm(sigmoid_product, ord=2))
print("||product of ReLU-like Jacobians||_2 ≈", np.linalg.norm(relu_product, ord=2))


# 17. Vectorized NumPy implementation

Below is a compact one-hidden-layer MLP with batch-vectorized backpropagation.


In [ ]:
import numpy as np

def sigmoid(z):
    return 1/(1+np.exp(-z))

def sigmoid_prime_from_activation(a):
    return a*(1-a)

class OneHiddenMLP:
    def __init__(self, d_in, d_hidden, d_out, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 0.2, size=(d_hidden, d_in))
        self.b1 = np.zeros((d_hidden, 1))
        self.W2 = rng.normal(0, 0.2, size=(d_out, d_hidden))
        self.b2 = np.zeros((d_out, 1))

    def forward(self, X):
        self.X = X
        self.Z1 = self.W1 @ X + self.b1
        self.A1 = sigmoid(self.Z1)
        self.Z2 = self.W2 @ self.A1 + self.b2
        self.A2 = sigmoid(self.Z2)
        return self.A2

    def loss(self, Yhat, Y):
        return 0.5 * np.mean((Yhat - Y)**2)

    def backward(self, Y):
        m = Y.shape[1]
        D2 = (self.A2 - Y) * sigmoid_prime_from_activation(self.A2)
        dW2 = (D2 @ self.A1.T) / m
        db2 = np.sum(D2, axis=1, keepdims=True) / m

        D1 = (self.W2.T @ D2) * sigmoid_prime_from_activation(self.A1)
        dW1 = (D1 @ self.X.T) / m
        db1 = np.sum(D1, axis=1, keepdims=True) / m
        return dW1, db1, dW2, db2

    def step(self, grads, lr=0.5):
        dW1, db1, dW2, db2 = grads
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        self.W2 -= lr * dW2
        self.b2 -= lr * db2

X = np.array([[0, 0, 1, 1],
              [0, 1, 0, 1]], dtype=float)
Y = np.array([[0, 1, 1, 0]], dtype=float)

model = OneHiddenMLP(d_in=2, d_hidden=4, d_out=1, seed=42)

for epoch in range(4000):
    Yhat = model.forward(X)
    J = model.loss(Yhat, Y)
    grads = model.backward(Y)
    model.step(grads, lr=1.0)
    if epoch % 500 == 0:
        print(f"epoch={epoch}, loss={J:.6f}")

print("\nFinal predictions:")
print(model.forward(X))


# 18. PyTorch autograd verification

This cell reproduces the worked example in PyTorch and compares autograd gradients to the manual ones.


In [ ]:
# Uncomment in Colab if needed:
# !pip install torch

import torch

torch.set_printoptions(precision=8, sci_mode=False)

x_t = torch.tensor([[1.0], [2.0]])
y_t = torch.tensor([[1.0]])

W1_t = torch.tensor([[0.1, 0.2],
                     [0.3, 0.4]], requires_grad=True)
b1_t = torch.tensor([[0.1],
                     [0.1]], requires_grad=True)

W2_t = torch.tensor([[0.5, 0.6]], requires_grad=True)
b2_t = torch.tensor([[0.1]], requires_grad=True)

z1_t = W1_t @ x_t + b1_t
a1_t = torch.sigmoid(z1_t)
z2_t = W2_t @ a1_t + b2_t
a2_t = torch.sigmoid(z2_t)
L_t = 0.5 * torch.sum((a2_t - y_t)**2)

L_t.backward()

print("Loss:", L_t.item())
print("\ndW1 =\n", W1_t.grad)
print("db1 =\n", b1_t.grad)
print("dW2 =\n", W2_t.grad)
print("db2 =\n", b2_t.grad)


# 19. Conceptual summary

Backpropagation can be understood at several levels:

1. **Intuition:** it assigns credit/blame to parameters.  
2. **Chain rule:** it repeatedly differentiates through intermediate variables.  
3. **Delta recursion:** it propagates $\delta^{[l]} = \partial L / \partial z^{[l]}$ backward.  
4. **Jacobian view:** it computes repeated vector-Jacobian products.  
5. **Autodiff view:** it is reverse-mode automatic differentiation on a computational graph.

These are all different descriptions of the same mechanism.


# 20. Key formulas to remember

For

$$
z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]},
\qquad
a^{[l]} = \phi^{[l]}(z^{[l]}),
$$

the backward formulas for one example are

$$
\delta^{[L]}
=
\frac{\partial L}{\partial a^{[L]}}
\odot
\phi^{[L]\,'}(z^{[L]}),
$$

$$
\delta^{[l]}
=
\left(W^{[l+1]\top}\delta^{[l+1]}\right)
\odot
\phi^{[l]\,'}(z^{[l]}),
$$

$$
\frac{\partial L}{\partial W^{[l]}}
=
\delta^{[l]} a^{[l-1]\top},
\qquad
\frac{\partial L}{\partial b^{[l]}}
=
\delta^{[l]}.
$$

For batches, replace vectors by activation matrices and average over the batch dimension.
